# 07 — DADPc (Domain Adaptive Deep Possibilistic Clustering)

**Family:** Domain adaptation &nbsp;•&nbsp; **Input:** DE features `[14 x 5]`

DADPc targets the **subject-independent** protocol. A shared feature extractor maps
DE features to an embedding; a linear head classifies the labelled source (22
training subjects). The cross-subject domain gap is reduced by a **possibilistic
clustering objective with a fuzzy-entropy regularizer** in embedding space, plus a
light multi-kernel MMD term:

`L = CE(source) + w_mmd·MMD(source, target) + w_pc·PCM(all) + w_fe·fuzzy_entropy(memberships)`

`target` = the **unlabelled** left-out subject (transductive domain adaptation —
its windows are visible during training, its labels never are).

## 1. Setup

In [ ]:
import os, sys

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src import benchmark_si as B
from src.evaluation import set_seed, get_device, count_parameters
from src.models import dadpc

set_seed(42)
DEVICE = get_device()

# ----- experiment configuration -----
MODEL_NAME = "DADPc"
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> 3-fold / few-epoch smoke test
                      # False -> the full 23-fold benchmark used for the thesis table
print("device:", DEVICE, "| model:", MODEL_NAME, "| label:", LABEL, "| QUICK:", QUICK)
print("recipe:", B.REGISTRY[MODEL_NAME])


## 2. Architecture

`src/models/dadpc.py`: `FeatureExtractor` (3-layer MLP with BN/dropout, 70 → 64-D
embedding) + linear classifier + `K = n_classes` learnable cluster centers with a
learnable per-cluster bandwidth `eta`. `memberships` implements the possibilistic
`u_ik = 1 / (1 + (d²/eta)^{1/(m-1)})` (not simplex-constrained). `dadpc_loss`
assembles the four terms. See the `# ASSUMPTION` notes in the source.

In [ ]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, 5)}
_m = dadpc.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

## 3. Run the subject-independent (LOSO) benchmark

DADPc is designed for LOSO only: it needs the held-out subject's **unlabelled**
windows during training (transductive domain adaptation). That bespoke loop —
target-minibatch sampling + `model.dadpc_loss` (CE + MMD + PCM + fuzzy entropy) —
lives in `src/benchmark_si.py::run_dadpc_loso`; metrics still flow through
`compute_metrics` / `summarize_per_fold` for schema parity with every other model.

`run_one` dispatches to it automatically from this model's `REGISTRY` entry
(`runner="dadpc"`), so the notebook just calls `run_one`.


In [ ]:
# One shared call: 23-fold transductive LOSO, then writes results/DADPc_results.json
res = B.run_one(MODEL_NAME, label=LABEL, quick=QUICK, device=DEVICE)
res

## Notes for the methodology chapter

* Loss weights `w_mmd / w_pc / w_fe` are the main knobs (set in this model's
  `src/benchmark_si.py::REGISTRY` entry) — a small sweep belongs in the appendix.
* Transductive setup: the left-out subject's raw windows (never labels) are used
  during training, which is standard for domain-adaptation EEG papers but should be
  stated explicitly in the thesis.
* Every metric comes from `src/evaluation.compute_metrics`; `params_count` and
  `train_time_sec` are logged to `results/{MODEL_NAME}_results.json`.
